In [14]:
import pandas as pd
import re


# DLW

load texts

In [15]:
import os

full_texts = {}
dir = "deeplearningweekly"
for file in os.listdir(dir):
    if file in ["associate-editor.txt", "want-to-see-your-content-in-our-next.txt", "open-source-deep-learning-curriculum.txt", 
                "733833_deep-learning-weekly-106.txt", # not same title format
                "deep-learning-weekly-ai-hardware.txt", "deep-learning-weekly-generative-modeling.txt", "deep-learning-weekly-self-supervised.txt", # special issues, different title format
                ]:
        continue # skip these special files
    assert "deep-learning" in file, file
    #print(file)
    with open(dir+"/"+file) as f:
        full_texts[file] = f.read()

for issue 425, i changed a paper title from ## to ### (line 114)

get the intersting parts

In [16]:
import re
title_count = {}
texts_by_date = {}


for file in full_texts:
    text = full_texts[file]

    # find date
    date = re.findall(r"\n[A-Z][a-z]{2} [0-3][0-9], 20[1-2][0-9]\n", text)
    assert len(date) == 1
    date=date[0][1:-1]

    # abstracts are not titles
    text = text.replace("\n\n## Abstract:\n\n", "\n\n#### Abstract:\n\n")
    text = text.replace("\n\n## Abstract\n\n", "\n\n#### Abstract:\n\n")


    # split into categories
    title_regex = r"\n## [^\n]*\n"

    # count category titles
    titles = re.findall(title_regex, text)
    for title in titles:

        if title in title_count:
            title_count[title] += 1
        else:
            title_count[title] = 1
    
    # split into entries
    titles_and_sections = re.split(r"\n(## [^\n]*\n)", text)
    titles_and_sections = titles_and_sections[1:] # remove start
    #print(len(titles_and_sections))
    for i in range(0,len(titles_and_sections),2):
        title = titles_and_sections[i]
        section = titles_and_sections[i+1]

        title = title[:-1]

        # prepare dict
        if not title in texts_by_date:
            texts_by_date[title]={}
        assert not date in texts_by_date[title]
        texts_by_date[title][date] = []


        assert "## " in title
        assert not "###" in title
        if not "### " in section:
            print("Not entry:", title, date)
            continue



        subtitles_and_entries = re.split(r"\n(### [^\n]*\n)", section)
        subtitles_and_entries = subtitles_and_entries[1:]
        #print(section)
        #for t in subtitles_and_entries:
        #    print(t)
        #    print("==========")
        #print("@@@@")
        for j in range(0, len(subtitles_and_entries), 2):
            subtitle = subtitles_and_entries[j]
            entry = subtitles_and_entries[j+1]
            entry = re.sub(r"\[Link\] [^\n]*\n", "", entry)
            assert "###" in subtitle
            assert not "\n### " in entry, entry
            texts_by_date[title][date].append(subtitle+entry)

import pprint
pprint.pprint(title_count)

{'\n## Code\n': 10,
 '\n## Datasets\n': 37,
 '\n## Fun\n': 5,
 '\n## Industry\n': 320,
 '\n## Learning\n': 320,
 '\n## Libraries & Code\n': 308,
 '\n## MLOps\n': 83,
 '\n## MLOps & LLMOps\n': 113,
 '\n## Mobile & Edge\n': 36,
 '\n## Mobile + Edge\n': 51,
 '\n## Mobile+Edge\n': 5,
 '\n## NeurIPS 2019\n': 1,
 '\n## Papers\n': 3,
 '\n## Papers & Publications\n': 317}


In [17]:
import pandas as pd

df_dlw = pd.DataFrame(columns=["date", "category", "text"])

for category in texts_by_date:
    for date in texts_by_date[category]:
        for text in texts_by_date[category][date]:
            df_dlw.loc[len(df_dlw)] = [date, category, text.replace("\n\n","\\n")] #\n\n seperates title from text. "\n" turn into " " by pandas so replacing this with "\\n" to keep it. Other "\n"s later in the text are replaced.


In [18]:
df_dlw["date"] = pd.to_datetime(df_dlw["date"])
#df_dlw

# bin by month

In [19]:
def group_dfs_in_months(m = 1): #m= number of months to combine in a bin (e.g. m=3 means we look at quarters)
    df_dlw.loc[:,"bin"] = ((df_dlw["date"].dt.month+m-1)//m)/100 + df_dlw["date"].dt.year


In [20]:
group_dfs_in_months()

In [21]:
df_dlw.groupby(["bin"]).count()


,date,category,text
bin,,,
2019.05,77,77,77
2019.06,59,59,59
2019.07,67,67,67
2019.08,61,61,61
2019.09,63,63,63
...,...,...,...
2025.06,54,54,54
2025.07,72,72,72
2025.08,64,64,64


# save

In [22]:
df_dlw.index = "dlw_"+df_dlw.index.astype(str)

In [ ]:
df_dlw[df_dlw["bin"]<2025.1][["date","text"]].to_csv("dlweekly_stories.csv")